In [ ]:
#imports
import pandas as pd
import os
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from wordcloud import WordCloud
import re

In [ ]:
# Select channel
channel = 'RTP'

#select date
date = 'Nov_10'


In [ ]:
# Load visual pkl for debate with person 1 and person 2 OR all debates with person 1 OR all debates
pklfiles = []

if channel and date:
    for file in os.listdir('Project_Features'):
        if file.endswith('ocr.pkl') and channel in file and date in file:
            pklfiles.append(file) 
elif channel and not date:
    for file in os.listdir('Project_Features'):
        if file.endswith('ocr.pkl') and channel in file:
            pklfiles.append(file)
elif date and not channel:
    for file in os.listdir('Project_Features'):
        if file.endswith('ocr.pkl') and date in file:
            pklfiles.append(file)
else:
    for file in os.listdir('Project_Features'):
        if file.endswith('ocr.pkl'):
            pklfiles.append(file)

data = []

for video in pklfiles:
    print(f"Loading: {video}")
    df = pd.read_pickle(os.path.join('Project_Features', video))
    data.append(df)

data = pd.concat(data, ignore_index=True)
#data = data.sort_values(by='Frame').reset_index(drop=True)

# Extract the video folder name (everything between 'Frames/' and '/frame_')
data['Video_Name'] = data['Frame'].str.extract(r'Frames/([^/]+)/')

# Extract the actual frame number and convert it to an integer
data['Frame_Number'] = data['Frame'].str.extract(r'frame_(\d+)\.jpg').astype(int)

# Sort first by the Video Name, then by the true integer Frame Number
data = data.sort_values(by=['Video_Name', 'Frame_Number']).reset_index(drop=True)

print('\n')
data.info()

all_text = []
for idx, frame in data.iterrows():
    frame_n = frame['Frame_Number']
    ocr = frame['OCR']

    if len(ocr) >0:
        for box in ocr:
            bbox = box['bbox']
            text = box['text']
            conf = box['conf']
    
            new_row = {'frame_n': frame_n,
                        'bbox': bbox,
                        'text': text,
                        'conf': conf}
            
            all_text.append(new_row)

df_ocr = pd.DataFrame(all_text)

print(df_ocr.head())
        


OCR collumn:
- List of dictionaries. 
- Dictionaries keys: 
    - 'bbox': A list of 4 numbers [x1, y1, x2, y2] representing the corners of the box drawn around the detected text.  
    - 'conf': Confidence in detection
    - 'text': Text detected

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df_ocr['conf'], bins=20, color='green', edgecolor='black')
plt.title('Histogram of OCR Confidences', fontsize=16)
plt.xlabel('Confidence Score', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(axis='y', alpha=0.75)
plt.show()

# quartiles of confidence scores
Q1 = np.percentile(df_ocr['conf'], 25)
Q3 = np.percentile(df_ocr['conf'], 75)

print(f"25th Percentile (Q1): {Q1:.4f}")
print(f"75th Percentile (Q3): {Q3:.4f}")

In [ ]:
#considering texts with confidence >= Q1
mask = df_ocr['conf'] >= Q1
df_conf_text = df_ocr[mask]
text_blob = ' '.join(df_conf_text['text'].dropna())
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text_blob)

plt.figure(figsize=(15, 7.5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of OCR Texts', fontsize=16)
plt.show() 

# -------- BAR CHART with 20 most common words

top_20 = df_conf_text['text'].value_counts().head(20)

plt.figure(figsize=(12, 6))
# top_20.index contains the actual words/phrases
# top_20.values contains the frequency numbers
plt.bar(top_20.index, top_20.values, color='skyblue')

plt.title('Top 20 Most Common OCR Texts', fontsize=16)
plt.xlabel('Texts', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.75)
plt.show()

## Remove Weird Symbols 

Create a mask to remove weird symbols (such as b and br) that were always appearing in the previous wordcloud and barchart

In [ ]:
df_cleaned = df_ocr.copy()

df_cleaned['text'] = df_cleaned['text'].str.replace(r'<b>|</b>|<br>', ' ', regex=True) #replace this weird thigs by ' '

df_cleaned = df_cleaned[df_cleaned['text'].str.len() > 1] #remove empty strings or strings with len=1 (havia b e 0)

text_blob_cleaned = ' '.join(df_cleaned['text'].dropna())

wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text_blob_cleaned)
plt.figure(figsize=(15, 7.5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of OCR Texts (Cleaned Weird Symbols)', fontsize=16)
plt.show() 

# -------- BAR CHART with 20 most common words

top_20 = df_cleaned['text'].value_counts().head(20)

plt.figure(figsize=(12, 6))
# top_20.index contains the actual words/phrases
# top_20.values contains the frequency numbers
plt.bar(top_20.index, top_20.values, color='skyblue')

plt.title('Top 20 Most Common OCR Texts (Cleaned Weird Symbols)', fontsize=16)
plt.xlabel('Texts', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.75)
plt.show()

## Remove Clocks and Logo

Create more masks to remove the clock and words (tvi, rtp, telejornal) because they are always on the screen, making the wordcloud enphasize  them

In [ ]:
#regex to look for specific pattern (number number : number number)
is_clock = df_cleaned['text'].str.contains(r'\d{1,2}:\d{2}', regex=True, na=False)

clocks_found = df_cleaned[is_clock]

print(f'Number of clocks caught: {len(clocks_found)}')
print("--- Clocks Caught ---")
print(clocks_found['text'].value_counts().head(3))
print('-----------------------------------------------------')


#keep everything that is NOT a clock
df_no_clock = df_cleaned[~is_clock]

is_logo = df_cleaned['text'].str.contains('rtp|tvi|cnn|direto|jornal', case=False, na=False)
df_no_logo = df_cleaned[~is_logo]

is_cleaner = ~(is_clock | is_logo)
df_no_clock_no_logo=df_cleaned[is_cleaner]


print("--- Top 10 Most Frequent Texts ---")
print(df_no_clock_no_logo['text'].value_counts().head(10))

#Wordcloud and bar graph

text_final_clean = ' '.join(df_no_clock_no_logo['text'].dropna())
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text_final_clean)

plt.figure(figsize=(15, 7.5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of OCR Texts (Clean Clock and Logos)', fontsize=16)
plt.show() 

# -------- BAR CHART with 20 most common words

top_20 = df_no_clock_no_logo['text'].value_counts().head(20)

plt.figure(figsize=(15, 7.5))
# top_20.index contains the actual words/phrases
# top_20.values contains the frequency numbers
plt.bar(top_20.index, top_20.values, color='skyblue')

plt.title('Top 20 Most Common OCR Texts (Clean Clock and Logos)', fontsize=16)
plt.xlabel('Texts', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.75)
plt.show()

# Bounding Box Location Analysis

In [ ]:
y_centers, x_centers = [], []

for box in df_cleaned['bbox']:
    if len(box)>0:
        x_c = (box[2]+box[0])/2
        y_c =(box[3]+box[1])/2
        x_centers.append(x_c)
        y_centers.append(y_c)

plt.figure(figsize=(10,6))
plt.scatter(x_centers, y_centers, alpha=0.5)
plt.grid(True)
plt.title('Centroid of Text Bounding Boxes (Raw)')
plt.ylabel('y center')
plt.xlabel('x center')


## How can the AI detect so much text everywhere????

Given the blob of points, we understand that not only the clock+logo+header+rodapé are being detected. More words are being detected, which means more bounding boxes are appearing everywhere.

In [ ]:
#add columns with centers
df_cleaned['x_center'] = (df_cleaned['bbox'].str[0] + df_cleaned['bbox'].str[2])/2
df_cleaned['y_center'] = (df_cleaned['bbox'].str[1] + df_cleaned['bbox'].str[3])/2
df_cleaned['area']= (df_cleaned['bbox'].str[3]- df_cleaned['bbox'].str[1])*(df_cleaned['bbox'].str[2]- df_cleaned['bbox'].str[0])

is_bottom = df_cleaned['y_center'] > 500 #this is so hardcoded help
is_area = df_cleaned['area'] > 12000
is_news = is_cleaner & is_bottom & is_area #filter this to include bigger areas which are more likely to be news text

df_news_zone = df_cleaned[is_news]

plt.figure(figsize=(12, 7))
plt.scatter(df_cleaned[is_news]['x_center'], df_cleaned[is_news]['y_center'], color='mediumseagreen', alpha=0.5, label='Actual News')
plt.scatter(df_cleaned[is_logo]['x_center'], df_cleaned[is_logo]['y_center'], color='royalblue', alpha=0.5, label='Logos')
plt.scatter(df_cleaned[is_clock]['x_center'], df_cleaned[is_clock]['y_center'], color='crimson', alpha=0.5, label='Clocks')

plt.title('Centroid of Bouding Boxes (filtered)', fontweight='bold')
plt.xlabel('X-Center')
plt.ylabel('Y-Center')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.gca().invert_yaxis() #had to invert

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df_cleaned['area'], bins=20, color='purple', edgecolor='black')
plt.title('Distribution of Bounding Box Areas', fontsize=16)
plt.xlabel('Area', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(axis='y', alpha=0.75)


In [ ]:
frame = 200

frame_id = data.iloc[frame]['Frame']
#print(frame_id)

img = Image.open(frame_id)

fig = plt.figure()
ax = fig.add_axes([0, 0, 1, 1])
ax.imshow(img)

print("\nDetected text:")

for t in data.iloc[frame]['OCR']:

    box = t['bbox']
    # Create a rectangle patch
    rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], linewidth=2, edgecolor='royalblue', facecolor='none')

    # Add patch to the image
    ax.add_patch(rect)

    print(f"{t['text']}; confidence {t['conf']}")


In [ ]:
# tamanho das caixas de texto